# Suite INS — Installs: schemas defined by transactions

A module defines its schemas in a module-scope `install(txn)` (docs/OBJECT_IDENTITY.md, Decided, 86;
[Installing schemas](../../docs/proposals/install/README.md)): its transaction upserts them, each schema object a symbol
of its meta-schema, named from the transaction (`txn.Schemas.OfObject`, 78), found by its name, else created, and its
spec written where it does not hold. Run again, an install writes nothing; another adds to what it finds. A schema with
no name of its own is defined in a spec where it is linked, `txn.Schemas.OfContainer(cb)`, which creates nothing.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Mutations, Paths, Plain, Schemas
from support import entries, raises


def crm(txn):
    """crm.Person, who knows persons since a year; crm.Shape, a union; crm.Grid, a container."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Knows": S.OfRelation, "Shape": S.OfUnion, "Grid": S.OfContainer})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]).properties({
                "since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies({
                "knows": lambda a: a.relation([_.Knows]).me("knower")}).adjacencies([
                lambda a: a.relation([_.Knows]).me("known")]),
            lambda _: _.Shape.name("crm.Shape").branches([
                lambda t: t.name("circle").schema([S.OfFloat]), lambda t: t.name("label").schema([S.OfStr])]),
            lambda _: _.Grid.name("crm.Grid").index([S.OfInt]).element([S.OfFloat]),
        ])]))])


def aged(txn):
    """crm.Person's age and a parameter, its era, added to what `crm` defined."""
    S = txn.Schemas
    return txn.uid("crm/2").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject})
        .spec([lambda _: _.Person.name("crm.Person").properties({"age": S.OfInt}).parameters({"era": lambda t: t})])]))])


def tagged(txn):
    """x.Item, whose tags are a list of text, a container with no name of its own, defined where it is linked."""
    S = txn.Schemas
    return txn.uid("tags/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Item": S.OfObject})
        .spec([lambda _: _.Item.name("x.Item").ref(True).properties({
            "tags": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr]))})])]))])

## INS-01 · A module's install defines its schemas, each kind a symbol of its meta-schema; run again, it writes nothing

In [ ]:
store = B.Store()
store.transaction(crm)
Person, Knows = store.registered("crm.Person"), store.registered("crm.Knows")
assert Person.ref and list(Person.properties) == ["name"] and Person.properties["name"].schema is store.Schemas.OfStr.as_schema()
assert list(Knows.entry.links) == ["knower", "known"] and list(Knows.entry.properties) == ["since"]
assert Person.adjacencies["knows"].relation is Knows and Person.adjacencies["knows"].me == "knower"
assert [b.name for b in store.registered("crm.Shape").branches] == ["circle", "label"]
grid = store.registered("crm.Grid")
assert grid.index is store.Schemas.OfInt.as_schema() and grid.element is store.Schemas.OfFloat.as_schema()
snapshot = Plain.ToPlain(store).Snapshot()
store.transaction(crm)  # run again: each found by its name, nothing written
assert Plain.ToPlain(store).Snapshot() == snapshot
with store.transaction():  # what it defined builds data
    ada = store.crm.Person(lambda c: c.name("Ada").knows(lambda a: a.insert(lambda e: e.known(lambda b: b.name("Bob")).since(2020))))
assert [str(e["known"].name) for e in entries(Person, ada, "knows")] == ["Bob"]

## INS-02 · A schema with no name of its own, defined in a spec where it is linked: made with the link, as the link's builder makes one, found holding when run again; linked twice, refused

In [ ]:
store = B.Store()
with store.transaction():
    outside = store.Schemas.OfContainer(lambda c: c.index([store.Schemas.OfInt]).element([store.Schemas.OfStr]))  # made
store.transaction(tagged)
tags = store.registered("x.Item").properties["tags"].schema
assert tags is not outside and tags.name is None  # defined where it is linked, made with the link
assert tags.index is store.Schemas.OfInt.as_schema() and tags.element is store.Schemas.OfStr.as_schema()
store.transaction(tagged)  # run again: an equal one is linked there, so nothing is made
assert store.registered("x.Item").properties["tags"].schema is tags


def twice(txn):
    S = txn.Schemas
    return txn.uid("twice/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Item": S.OfObject})
        .spec([lambda _: (lambda texts: _.Item.name("x.Twice").properties({"a": texts, "b": texts}))(
            S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr])))])]))])


with raises(ValueError, match="an object a spec defines is linked where it is defined"):
    store.transaction(twice)  # one definition, linked from one place


def required(txn):
    S = txn.Schemas
    return txn.uid("required/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Rule": S.OfPattern})
        .spec([lambda _: _.Rule.name("x.Rule").requires([lambda c: c.description("holds")])])]))])


store.transaction(required)  # a pattern's constraint is made as its builder makes one: its result, Bool
assert store.registered("x.Rule").requires[0].result is store.Schemas.OfBool.as_schema()

## INS-03 · An install adds to what another defined; a transaction given to an install names its own members, and the store's schemas besides

In [ ]:
store = B.Store()
store.transaction(crm)
store.transaction(aged)
assert list(store.registered("crm.Person").properties) == ["name", "age"]
assert list(store.registered("crm.Person").parameters) == ["era"]  # an entry given nothing defines what it links


def paired(txn):
    S = txn.Schemas
    return txn.uid("pair/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Pair": S.OfUnion})
        .spec([lambda _: _.Pair.name("x.Pair").branches([lambda t: t.name("n").schema([S.OfInt, S.OfStr])])])]))])


with raises(TypeError, match="link 'schema' links one schema, got 2"):
    store.transaction(paired)


def units(txn):
    """x.Pairs, its entries a set of `a`, given as a builder takes it; and a unit on every object schema."""
    S = txn.Schemas
    return txn.uid("units/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Pairs": S.OfRelation, "Object": S.OfObject})
        .spec([
            lambda _: _.Pairs.name("x.Pairs").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques("a"),
            lambda _: _.Object.name("Schemas.OfObject").properties({"unit": S.OfStr}),
        ])]))])


def meters(txn):
    """x.Meters, its unit the one `units` added."""
    return txn.uid("meters/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Meters": txn.Schemas.OfObject})
        .spec([lambda _: _.Meters.name("x.Meters").unit("m").description(lambda d: d.as_value("metres"))])]))])  # given whole, or visited


store.transaction(units)
store.transaction(meters)
assert store.registered("x.Pairs").uniques == (frozenset({"a"}),)  # a spec takes a builder's shorthand
assert B.OfObject.Builder(store.registered("x.Meters"), False, store).unit.get() == "m" and store.registered("x.Meters").description == "metres"


def named(txn):
    """crm.Named, a pattern over crm.Person, which `crm` installed, its spec recorded over it."""
    S = txn.Schemas
    return txn.uid("named/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Named": S.OfPattern})
        .spec([lambda _: _.Named.name("crm.Named").symbols([lambda t: t.name("a").schema([txn.crm.Person])]).spec(
            lambda _: _.a.name("Ada"))])]))])  # one callback, or a list of them


def unready(txn):
    """x.Over, a pattern over x.Thing, which the same install defines: not yet there when its spec is recorded."""
    S = txn.Schemas
    return txn.uid("unready/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Thing": S.OfObject, "Over": S.OfPattern})
        .spec([lambda _: _.Over.name("x.Over").symbols([lambda t: t.name("a").schema([_.Thing])]).spec([lambda _: _.a])])]))])


def untyped(txn):
    """x.Bare, a pattern whose symbol names no schema."""
    return txn.uid("untyped/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Bare": txn.Schemas.OfPattern})
        .spec([lambda _: _.Bare.name("x.Bare").symbols([lambda t: t.name("a")]).spec([lambda _: _.a])])]))])


def inline(txn):
    """x.Inline, a pattern whose symbol's schema is defined where it is linked."""
    S = txn.Schemas
    return txn.uid("inline/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Inline": S.OfPattern})
        .spec([lambda _: _.Inline.name("x.Inline").symbols([lambda t: t.name("a").schema([S.OfObject(lambda o: o)])]).spec([
            lambda _: _.a])])]))])


store.transaction(named)
store.transaction(named)  # run again: the spec it gives holds
assert len(store.registered("crm.Named").spec) == 1 and list(store.registered("crm.Named").spec[0]) == ["a"]
for install in (unready, untyped, inline):
    with raises(ValueError, match="symbol 'a' of a pattern whose spec a spec records names no schema the store holds"):
        store.transaction(install)
seen = []


def looking(txn):
    seen.append((txn.Schemas.OfInt.as_schema() is store.Schemas.OfInt.as_schema(), repr(txn)))
    with raises(AttributeError):
        txn._hidden
    return txn.uid("look/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Int": txn.Schemas.OfScalar}).spec([lambda _: _.Int.name("Schemas.OfInt")])]))])  # the store's own, found


store.transaction(looking)
assert seen == [(True, "<transaction>")]

## INS-04 · The framework installs itself: a store is made by reading the installs of Schemas, Paths and Mutations; each, run again as a transaction, writes nothing

In [ ]:
store = B.Store()  # made by the bootstrap, reading each install through the builders
assert store.registered("Mutations.OfProperty.Set").properties["value"].schema is store.registered("Mutations.Payload")
assert store.registered("Schemas.OfObject").adjacencies["properties"].relation is store.registered("Schemas.HasProperty")
held = {name: store.registered(name) for name in store.names()}
for module in (Schemas, Paths, Mutations):
    store.transaction(module.install)  # each found by its name; one it would change is refused (the store's own)
assert {name: store.registered(name) for name in store.names()} == held

## INS-05 · An install removes members of the store's own schemas, each entry by its name: a property, an adjacency, a branch; run again, it writes nothing; a schema the store was made holding refuses it

In [ ]:
store = B.Store()
store.transaction(crm)
store.transaction(aged)


def trimmed(txn):
    """crm.Person without its age and its knows, crm.Shape without its label: each entry removed by its name."""
    S = txn.Schemas
    return txn.uid("crm/3").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Person": S.OfObject, "Shape": S.OfUnion})
        .spec([
            lambda _: _.Person.name("crm.Person").properties(lambda a: a.remove(lambda e: e.name("age")))
            .adjacencies(lambda a: a.remove(lambda e: e.name("knows"))),
            lambda _: _.Shape.name("crm.Shape").branches(lambda a: a.remove(lambda e: e.name("label"))),
        ])]))])


store.transaction(trimmed)
Person = store.registered("crm.Person")
assert list(Person.properties) == ["name"] and list(Person.adjacencies) == [] and len(Person.anonymous) == 1
assert [b.name for b in store.registered("crm.Shape").branches] == ["circle"] and Person.validate() == []
snapshot = Plain.ToPlain(store).Snapshot()
store.transaction(trimmed)  # run again: what it removes is absent, so it holds, and nothing is written
assert Plain.ToPlain(store).Snapshot() == snapshot


def unref(name):
    """A member of Schemas.OfObject, a schema the store was made holding, removed."""
    return lambda txn: txn.uid("unref/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Object": txn.Schemas.OfObject})
        .spec([lambda _: _.Object.name("Schemas.OfObject").properties(lambda a: a.remove(lambda e: e.name(name)))])]))])


with raises(TypeError, match="an extension only adds members to 'Schemas.OfObject': it removes or replaces what 'properties' holds"):
    store.transaction(unref("ref"))
store.transaction(unref("nope"))  # absent: its spec holds, and nothing is written
assert "ref" in store.registered("Schemas.OfObject").properties